In [38]:
from typing import Literal
import pandas as pd

from model_xray.configs.enums import *

from model_xray.utils.script_utils import get_siamese_results_dataframe

# mc_name:Literal['famous_le_10m', 'famous_le_100m']="famous_le_10m"
# imtype:ImageType=ImageType.GRAYSCALE_FOURPART
# imsize:int=256
# mode:Literal['st', 'es', 'ub', 'none']='ub'

# model_arch: Literal['osl_siamese_cnn', 'srnet'] = 'srnet'
# embed_payload_type: PayloadType = PayloadType.BINARY_FILE

def ret_full_df(df_infos: list[dict[str, str]]) -> pd.DataFrame:

    dfs = []
    for df_info in df_infos:

        df = get_siamese_results_dataframe(
            mc_name=df_info["mc_name"],
            imtype=df_info["imtype"],
            imsize=df_info["imsize"],
            mode=df_info["mode"],
            model_arch=df_info["model_arch"],
            embed_payload_type=df_info["embed_payload_type"],
        )

        df.rename(columns={'run num':'run_num', 'mc': 'test_mc', 'lsb': 'lsb'}, inplace=True)

        df['train_mc'] = df_info['mc_name']
        df['imtype'] = df_info["imtype"]
        df['imsize'] = df_info["imsize"]
        df['mode'] = df_info["mode"]
        df['embed_payload_type'] = df_info["embed_payload_type"]

        dfs.append(df)

    df = pd.concat(dfs, ignore_index=True)
    return df

from visualize_utils import calc_metrics
from functools import partial
calc_only_weighted = partial(calc_metrics, centroid=True, nn=True, weighted=True, unweighted=False)

def ret_df_w_weighted_metric(df):
    df_w_metrics = df[(df['test_mc']!='maleficnet_benigns') & (df['test_mc']!='maleficnet_mals')].groupby(
        ['run_num','test_mc', 'model_lsb', 'imtype', 'imsize', 'mode', 'embed_payload_type', 'train_mc', 'model_sha256', 'model_arch']
        ).apply(calc_only_weighted, include_groups=False,)

    df_w_metrics = df_w_metrics.reset_index()
    return df_w_metrics


df_infos = [
{
    "mc_name": "famous_le_10m",
    "imtype": ImageType.GRAYSCALE_FOURPART,
    "imsize": 256,
    "mode": "ub",
    "model_arch": "srnet",
    "embed_payload_type": PayloadType.BINARY_FILE
},
{
    "mc_name": "famous_le_10m",
    "imtype": ImageType.GRAYSCALE_FOURPART,
    "imsize": 100,
    "mode": "ub",
    "model_arch": "osl_siamese_cnn",
    "embed_payload_type": PayloadType.BINARY_FILE
},
{
    "mc_name": "famous_le_100m",
    "imtype": ImageType.GRAYSCALE_FOURPART,
    "imsize": 256,
    "mode": "ub",
    "model_arch": "srnet",
    "embed_payload_type": PayloadType.BINARY_FILE
},
{
    "mc_name": "famous_le_100m",
    "imtype": ImageType.GRAYSCALE_FOURPART,
    "imsize": 100,
    "mode": "ub",
    "model_arch": "osl_siamese_cnn",
    "embed_payload_type": PayloadType.BINARY_FILE
},
]

df = ret_full_df(df_infos)
df.head()

,run_num,test_mc,lsb,test_acc_centroid,test_acc_nn,model_lsb,model_arch,model_sha256,train_mc,imtype,imsize,mode,embed_payload_type
0,0,torch_pretrained_classification,0,0.675,0.675,1,srnet,8a6560a44332b6d259ebdd2c0371a3c0167ef0ef6c91b5...,famous_le_10m,grayscale_fourpart,256,ub,binary_file
1,0,famous_le_10m,0,0.600,0.400,1,srnet,8a6560a44332b6d259ebdd2c0371a3c0167ef0ef6c91b5...,famous_le_10m,grayscale_fourpart,256,ub,binary_file
2,0,famous_le_10m,1,0.400,0.600,1,srnet,8a6560a44332b6d259ebdd2c0371a3c0167ef0ef6c91b5...,famous_le_10m,grayscale_fourpart,256,ub,binary_file
3,0,famous_le_10m,2,0.400,0.600,1,srnet,8a6560a44332b6d259ebdd2c0371a3c0167ef0ef6c91b5...,famous_le_10m,grayscale_fourpart,256,ub,binary_file
4,0,famous_le_10m,3,0.400,0.600,1,srnet,8a6560a44332b6d259ebdd2c0371a3c0167ef0ef6c91b5...,famous_le_10m,grayscale_fourpart,256,ub,binary_file


In [39]:
df_w_weighted_metric = ret_df_w_weighted_metric(df)
df_w_weighted_metric.head()

,run_num,test_mc,model_lsb,imtype,imsize,mode,embed_payload_type,train_mc,model_sha256,model_arch,weighted_metric_centroid,weighted_metric_nn
0,0,famous_le_100m,1,grayscale_fourpart,100,ub,binary_file,famous_le_100m,db5f88b83a794f3c82f7afe0fabf42be2da7196ec9e8f5...,osl_siamese_cnn,0.443970,0.386387
1,0,famous_le_100m,1,grayscale_fourpart,100,ub,binary_file,famous_le_10m,00073b9360b2de6b0b58a702dab78311f603c4af42b63d...,osl_siamese_cnn,0.504529,0.435688
2,0,famous_le_100m,1,grayscale_fourpart,256,ub,binary_file,famous_le_100m,c02b88e9f85326f620de08ba4c5ead767832ad6a87844a...,srnet,0.619565,0.595756
3,0,famous_le_100m,1,grayscale_fourpart,256,ub,binary_file,famous_le_10m,8a6560a44332b6d259ebdd2c0371a3c0167ef0ef6c91b5...,srnet,0.684136,0.689829
4,0,famous_le_100m,2,grayscale_fourpart,100,ub,binary_file,famous_le_100m,bcc20ed19d40f0923bba583ea363629a51eeb74a4d46d0...,osl_siamese_cnn,0.520445,0.570393


In [49]:
def get_best_model(mc, df, eval_type:Literal['centroid', 'nn'] = 'nn') -> "sha256string":
    sha256 = df.iloc[df[(df['test_mc'] == mc)][f'weighted_metric_{eval_type}'].idxmax()]['model_sha256']
    return sha256

best_model = get_best_model('famous_le_100m', df_w_weighted_metric, 'centroid')

In [50]:
def get_best_model_results(model_sha256, df):
    return df[df['model_sha256'] == model_sha256].copy()

def strip_nonunique_cols(df):
    cols_to_keep = [col for col in df.columns if len(df[col].unique()) > 1]
    return df[cols_to_keep]

df_bm = strip_nonunique_cols(get_best_model_results(best_model, df))
df_bm

,test_mc,lsb,test_acc_centroid,test_acc_nn
9065,torch_pretrained_classification,0,0.925000,0.687500
9066,famous_le_10m,0,1.000000,0.600000
9067,famous_le_10m,1,0.000000,0.600000
9068,famous_le_10m,2,0.000000,0.600000
9069,famous_le_10m,3,0.200000,0.800000
9070,famous_le_10m,4,0.200000,1.000000
9071,famous_le_10m,5,0.400000,1.000000
9072,famous_le_10m,6,1.000000,1.000000
9073,famous_le_10m,7,1.000000,1.000000
9074,famous_le_10m,8,1.000000,1.000000
